# 1.2 Column Selection and Modification ? Oracle HR Practice

**Hands-on PySpark notebook | Oracle HR sample-schema domain**

All records are synthetic and follow Oracle HR table conventions. The reusable HR DataFrames
include valid keys, NULLs, a missing department assignment, and duplicate employee email data
for practical migration and quality-control exercises.

**Learning objectives**
- Prepare HR projections with clear column names and stable output contracts.
- Add and remove columns while preserving data types, NULL behavior, and source records.
- Compose reusable DataFrame transformations for workforce data pipelines.
- Validate expected schemas, row counts, and HR data-quality edge cases.

## 1. PySpark environment setup

Target: PySpark 3.5+ with a compatible Java runtime. All exercises use the Oracle HR sample
schema DataFrames; Pandas is not used.

## 2. Required imports

In [ ]:
from datetime import date
from decimal import Decimal
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import DateType, DecimalType, IntegerType, StringType, StructField, StructType

## 3. SparkSession creation

In [ ]:
spark = (SparkSession.builder.appName("Oracle HR DataFrame Creation and Structure Practice")
         .master("local[2]").getOrCreate())
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

## 4. Create reusable Oracle HR DataFrames

The EMPLOYEES and DEPARTMENTS tables contain 20 and 18 records. Related JOBS, JOB_HISTORY,
LOCATIONS, COUNTRIES, and REGIONS records are included. Primary keys are unique and present
foreign keys point to corresponding records; NULL optional relationships are intentional.

In [ ]:
region_schema = StructType([StructField("region_id", IntegerType(), False), StructField("region_name", StringType(), False)])
regions_df = spark.createDataFrame([
    (1, "Europe"), (2, "Americas"), (3, "Asia"), (4, "Middle East and Africa"), (5, "Oceania")
], region_schema)

country_schema = StructType([
    StructField("country_id", StringType(), False), StructField("country_name", StringType(), False),
    StructField("region_id", IntegerType(), False)
])
countries_df = spark.createDataFrame([
    ("US", "United States of America", 2), ("CA", "Canada", 2), ("BR", "Brazil", 2),
    ("GB", "United Kingdom", 1), ("DE", "Germany", 1), ("FR", "France", 1),
    ("IN", "India", 3), ("JP", "Japan", 3), ("AU", "Australia", 5), ("EG", "Egypt", 4)
], country_schema)

location_schema = StructType([
    StructField("location_id", IntegerType(), False), StructField("street_address", StringType(), True),
    StructField("postal_code", StringType(), True), StructField("city", StringType(), False),
    StructField("state_province", StringType(), True), StructField("country_id", StringType(), False)
])
locations_df = spark.createDataFrame([
    (1000, "200 Market Street", "94105", "San Francisco", "California", "US"),
    (1001, "500 Fifth Avenue", "10018", "New York", "New York", "US"),
    (1002, "77 Massachusetts Avenue", "02139", "Cambridge", "Massachusetts", "US"),
    (1003, "10 King Street", "M5H 1A1", "Toronto", "Ontario", "CA"),
    (1004, "120 Rua Augusta", "01305-000", "Sao Paulo", "Sao Paulo", "BR"),
    (1005, "25 Canada Square", "E14 5LQ", "London", None, "GB"),
    (1006, "Unter den Linden 5", "10117", "Berlin", None, "DE"),
    (1007, "8 Rue de Rivoli", "75001", "Paris", None, "FR"),
    (1008, "14 Residency Road", "560025", "Bengaluru", "Karnataka", "IN"),
    (1009, "6 Connaught Place", "110001", "New Delhi", "Delhi", "IN"),
    (1010, "33 Marine Drive", "400020", "Mumbai", "Maharashtra", "IN"),
    (1011, "1 Chome Marunouchi", "100-0005", "Tokyo", None, "JP"),
    (1012, "88 George Street", "2000", "Sydney", "New South Wales", "AU"),
    (1013, "22 Nile Corniche", None, "Cairo", None, "EG"),
    (1014, "400 Lake Shore Drive", "60611", "Chicago", "Illinois", "US"),
    (1015, "9 Market Street", "94105", "San Francisco", "California", "US"),
    (1016, "70 Fleet Street", "EC4Y 1EU", "London", None, "GB"),
    (1017, "15 Park Street", None, "Kolkata", "West Bengal", "IN")
], location_schema)

job_schema = StructType([
    StructField("job_id", StringType(), False), StructField("job_title", StringType(), False),
    StructField("min_salary", DecimalType(8, 2), False), StructField("max_salary", DecimalType(8, 2), False)
])
jobs_df = spark.createDataFrame([
    ("AD_PRES", "President", Decimal("20000.00"), Decimal("40000.00")),
    ("AD_VP", "Administration Vice President", Decimal("15000.00"), Decimal("30000.00")),
    ("IT_PROG", "Programmer", Decimal("4000.00"), Decimal("12000.00")),
    ("FI_MGR", "Finance Manager", Decimal("8000.00"), Decimal("16000.00")),
    ("FI_ACCOUNT", "Accountant", Decimal("4000.00"), Decimal("10000.00")),
    ("PU_MAN", "Purchasing Manager", Decimal("8000.00"), Decimal("15000.00")),
    ("PU_CLERK", "Purchasing Clerk", Decimal("2000.00"), Decimal("5000.00")),
    ("SA_MAN", "Sales Manager", Decimal("10000.00"), Decimal("20000.00")),
    ("SA_REP", "Sales Representative", Decimal("5000.00"), Decimal("12000.00")),
    ("HR_REP", "Human Resources Representative", Decimal("4000.00"), Decimal("9000.00")),
    ("HR_MGR", "Human Resources Manager", Decimal("8000.00"), Decimal("15000.00")),
    ("MK_MAN", "Marketing Manager", Decimal("9000.00"), Decimal("15000.00")),
    ("MK_REP", "Marketing Representative", Decimal("5000.00"), Decimal("11000.00")),
    ("ST_MAN", "Stock Manager", Decimal("5000.00"), Decimal("10000.00")),
    ("ST_CLERK", "Stock Clerk", Decimal("2000.00"), Decimal("5000.00")),
    ("AC_MGR", "Accounting Manager", Decimal("8000.00"), Decimal("16000.00")),
    ("AC_ACCOUNT", "Public Accountant", Decimal("4000.00"), Decimal("10000.00")),
    ("PR_REP", "Public Relations Representative", Decimal("5000.00"), Decimal("11000.00"))
], job_schema)

employee_schema = StructType([
    StructField("employee_id", IntegerType(), False), StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), False), StructField("email", StringType(), True),
    StructField("phone_number", StringType(), True), StructField("hire_date", DateType(), False),
    StructField("job_id", StringType(), False), StructField("salary", DecimalType(8, 2), False),
    StructField("commission_pct", DecimalType(4, 2), True), StructField("manager_id", IntegerType(), True),
    StructField("department_id", IntegerType(), True)
])
employees_df = spark.createDataFrame([
    (100, "Olivia", "Bennett", "OBENNETT", "515.123.1000", date(1987,6,17), "AD_PRES", Decimal("24000.00"), None, None, 90),
    (101, "Marcus", "Chen", "MCHEN", "515.123.1001", date(1989,9,21), "AD_VP", Decimal("17000.00"), None, 100, 90),
    (102, "Priya", "Nair", "PNAIR", "515.123.1002", date(1993,1,13), "AD_VP", Decimal("17000.00"), None, 100, 90),
    (103, "Daniel", "Brooks", "DBROOKS", "590.423.1003", date(1990,1,3), "IT_PROG", Decimal("9000.00"), None, 102, 60),
    (104, "Sofia", "Patel", "SPATEL", "590.423.1004", date(1991,5,21), "IT_PROG", Decimal("6000.00"), None, 103, 60),
    (105, "Ethan", "Morris", "EMORRIS", "590.423.1005", date(1997,6,25), "IT_PROG", Decimal("4800.00"), None, 103, 60),
    (106, "Aisha", "Khan", "AKHAN", "590.423.1006", date(1998,2,5), "IT_PROG", Decimal("4800.00"), None, 103, 60),
    (107, "Lucas", "Meyer", "LMEYER", "590.423.1007", date(1999,2,7), "IT_PROG", Decimal("4200.00"), None, 103, 60),
    (108, "Grace", "Kim", "GKIM", "515.124.1008", date(1994,8,17), "FI_MGR", Decimal("12000.00"), None, 101, 100),
    (109, "Noah", "Williams", "NWILLIAMS", "515.124.1009", date(1994,8,16), "FI_ACCOUNT", Decimal("9000.00"), None, 108, 100),
    (110, "Elena", "Garcia", "EGARCIA", "515.124.1010", date(1997,9,28), "FI_ACCOUNT", Decimal("8200.00"), None, 108, 100),
    (111, "Ibrahim", "Hassan", "IHASSAN", "515.124.1011", date(1997,9,30), "FI_ACCOUNT", Decimal("7700.00"), None, 108, 100),
    (112, "Chloe", "Dubois", "CDUBOIS", "515.124.1012", date(1998,3,7), "FI_ACCOUNT", Decimal("7800.00"), None, 108, 100),
    (113, "Mateo", "Silva", "MSILVA", "515.124.1013", date(1999,12,7), "FI_ACCOUNT", Decimal("6900.00"), None, 108, 100),
    (114, "Amara", "Okafor", "AOKAFOR", "515.127.1014", date(1994,12,7), "PU_MAN", Decimal("11000.00"), None, 100, 30),
    (115, "Theo", "Martin", "TMARTIN", "515.127.1015", date(1995,5,18), "PU_CLERK", Decimal("3100.00"), None, 114, 30),
    (116, "Mei", "Tanaka", "MTANAKA", "515.127.1016", date(1997,12,24), "SA_MAN", Decimal("11000.00"), Decimal("0.15"), 100, 80),
    (117, "Fatima", "Rahman", "FRAHMAN", "515.127.1017", date(1997,7,24), "SA_REP", Decimal("8000.00"), Decimal("0.10"), 116, 80),
    (118, "Fatima", "Rahman", "FRAHMAN", "515.127.1018", date(1997,7,24), "SA_REP", Decimal("8000.00"), Decimal("0.10"), 116, 80),
    (119, "Ben", "Carter", None, None, date(1999,8,10), "HR_REP", Decimal("2500.00"), None, 100, None)
], employee_schema)

department_schema = StructType([
    StructField("department_id", IntegerType(), False), StructField("department_name", StringType(), False),
    StructField("manager_id", IntegerType(), True), StructField("location_id", IntegerType(), True)
])
departments_df = spark.createDataFrame([
    (10,"Administration",101,1000), (20,"Marketing",103,1001), (30,"Purchasing",114,1003),
    (40,"Human Resources",119,1008), (50,"Shipping",114,1009), (60,"IT",103,1002),
    (70,"Public Relations",101,1005), (80,"Sales",116,1014), (90,"Executive",100,1000),
    (100,"Finance",108,1006), (110,"Accounting Operations",109,1007), (120,"Recruiting",119,1017),
    (130,"Compensation",101,1008), (140,"Business Intelligence",103,1015),
    (150,"Customer Support",114,1016), (160,"Risk Management",108,1001),
    (170,"Legal",101,1005), (180,"Facilities",None,None)
], department_schema)

job_history_schema = StructType([
    StructField("employee_id", IntegerType(), False), StructField("start_date", DateType(), False),
    StructField("end_date", DateType(), False), StructField("job_id", StringType(), False),
    StructField("department_id", IntegerType(), True)
])
job_history_df = spark.createDataFrame([
    (101,date(1987,9,21),date(1989,9,20),"AC_ACCOUNT",110),
    (102,date(1989,1,13),date(1992,12,31),"IT_PROG",60),
    (103,date(1987,1,3),date(1989,12,31),"SA_REP",80),
    (104,date(1989,5,21),date(1991,5,20),"IT_PROG",60),
    (105,date(1994,6,25),date(1997,6,24),"IT_PROG",60),
    (108,date(1990,8,17),date(1994,8,16),"FI_ACCOUNT",100),
    (109,date(1990,8,16),date(1994,8,15),"AC_ACCOUNT",110),
    (110,date(1994,9,28),date(1997,9,27),"AC_ACCOUNT",110),
    (114,date(1990,12,7),date(1994,12,6),"PU_CLERK",30),
    (115,date(1992,5,18),date(1995,5,17),"PU_CLERK",30),
    (116,date(1993,12,24),date(1997,12,23),"SA_REP",80),
    (117,date(1994,7,24),date(1997,7,23),"SA_REP",80),
    (118,date(1994,7,24),date(1997,7,23),"SA_REP",80),
    (119,date(1995,8,10),date(1999,8,9),"HR_REP",40),
    (101,date(1985,1,1),date(1987,9,20),"AC_ACCOUNT",110)
], job_history_schema)

hr_dataframes = {
    "employees_df": employees_df, "departments_df": departments_df,
    "jobs_df": jobs_df, "job_history_df": job_history_df,
    "locations_df": locations_df, "countries_df": countries_df, "regions_df": regions_df
}
expected_counts = {"employees_df":20, "departments_df":18, "jobs_df":18,
                   "job_history_df":15, "locations_df":18, "countries_df":10, "regions_df":5}
for dataframe_name, dataframe in hr_dataframes.items():
    count = dataframe.count()
    assert count == expected_counts[dataframe_name], f"{dataframe_name}: unexpected record count {count}"
    print(f"{dataframe_name}: {count} records")

employee_ids = {row.employee_id for row in employees_df.select("employee_id").collect()}
department_ids = {row.department_id for row in departments_df.select("department_id").collect()}
job_ids = {row.job_id for row in jobs_df.select("job_id").collect()}
location_ids = {row.location_id for row in locations_df.select("location_id").collect()}
country_ids = {row.country_id for row in countries_df.select("country_id").collect()}
region_ids = {row.region_id for row in regions_df.select("region_id").collect()}
assert employees_df.select("employee_id").distinct().count() == 20
assert departments_df.select("department_id").distinct().count() == 18
assert all(r.manager_id is None or r.manager_id in employee_ids for r in employees_df.select("manager_id").collect())
assert all(r.manager_id is None or r.manager_id in employee_ids for r in departments_df.select("manager_id").collect())
assert all(r.department_id is None or r.department_id in department_ids for r in employees_df.select("department_id").collect())
assert all(r.job_id in job_ids for r in employees_df.select("job_id").collect())
assert all(r.employee_id in employee_ids and r.job_id in job_ids and (r.department_id is None or r.department_id in department_ids) for r in job_history_df.collect())
assert all(r.country_id in country_ids for r in locations_df.select("country_id").collect())
assert all(r.region_id in region_ids for r in countries_df.select("region_id").collect())
assert all(r.location_id is None or r.location_id in location_ids for r in departments_df.select("location_id").collect())
print("Primary/foreign key checks passed; duplicate employee email is intentional.")

## 5. Preview sample records and schemas

Review sample records and declared schemas before attempting the exercises. These same seven
DataFrames are reused throughout the notebook.

In [ ]:
for dataframe_name, dataframe in hr_dataframes.items():
    print(f"\n--- {dataframe_name}: sample rows ---")
    dataframe.show(3, truncate=False)
    print(f"--- {dataframe_name}: schema ---")
    dataframe.printSchema()

## 6. Practical questions

Each question is one Markdown prompt followed by one completely empty code cell. There are three
questions for each of the ten listed concepts.

## select()

### Question 1 ? Easy

**Business scenario:** The HR operations team needs a lightweight employee roster for a department directory.

**Problem statement:** Use `select()` on `employees_df` to return employee ID, first name, last name, job ID, and department ID in that order.

**Expected output:** A 20-row, five-column DataFrame with the requested fields; employee 119 remains present with a NULL department ID.

### Question 2 ? Medium

**Business scenario:** Workforce reporting needs a focused list of employees in the IT department.

**Problem statement:** Filter `employees_df` to department 60 and use `select()` to return employee ID, employee name fields, hire date, job ID, and salary. Validate the output row count.

**Expected output:** Five employees are returned with columns `employee_id`, `first_name`, `last_name`, `hire_date`, `job_id`, and `salary`; salaries range from 4200.00 to 9000.00.

### Question 3 ? Hard

**Business scenario:** HR master-data cleansing must preserve records that have incomplete reporting relationships or duplicate business identifiers.

**Problem statement:** Use `select()` to create a quality-review projection containing employee ID, email, manager ID, department ID, and commission percentage. Validate row count and identify the repeated email and NULL fields without removing records.

**Expected output:** All 20 employees remain in five columns; email FRAHMAN appears for employees 117 and 118, employee 100 has no manager, employee 119 has no department, and non-sales employees have NULL commission percentages.

## selectExpr()

### Question 4 ? Easy

**Business scenario:** A workforce cost report requires annualized compensation based on the monthly salary stored in EMPLOYEES.

**Problem statement:** Use `selectExpr()` to return employee ID, job ID, monthly salary, and an `annual_salary` expression equal to salary multiplied by 12.

**Expected output:** The output has 20 rows and four columns; employee 100 annualizes to 288000.00 and salary remains represented as a numeric decimal value.

### Question 5 ? Medium

**Business scenario:** An HR migration extract must standardize the employee name display and expose compensation with a stable report alias.

**Problem statement:** Use `selectExpr()` to return employee ID, a concatenated full name as `employee_name`, department ID, and salary as `monthly_salary`. Preserve NULL department values.

**Expected output:** The four output columns are `employee_id`, `employee_name`, `department_id`, and `monthly_salary`; all 20 rows remain and employee 119 retains a NULL department ID.

### Question 6 ? Hard

**Business scenario:** Compensation governance needs a row-level classification that separates missing commission data from commission-bearing employees.

**Problem statement:** Use `selectExpr()` with CASE logic to return employee ID, job ID, commission percentage, and `commission_category`: `NO_COMMISSION` when the commission is NULL, `STANDARD_COMMISSION` when it is at most 0.10, and `ELEVATED_COMMISSION` otherwise.

**Expected output:** All 20 employees are represented; 17 are `NO_COMMISSION`, employees 117 and 118 are `STANDARD_COMMISSION`, and employee 116 is `ELEVATED_COMMISSION`.

## col()

### Question 7 ? Easy

**Business scenario:** The HR help desk needs to find employee records that are not assigned to a department.

**Problem statement:** Use `col()` in a filter on `employees_df` to return employee ID, first name, last name, and department ID for employees with a NULL department.

**Expected output:** One row is returned: employee 119, with a NULL department ID.

### Question 8 ? Medium

**Business scenario:** A management hierarchy extract must isolate top-level employees and employees with missing department assignments.

**Problem statement:** Use `col()` expressions and NULL-safe predicates to return employees whose manager ID is NULL or whose department ID is NULL. Include employee ID, manager ID, and department ID.

**Expected output:** Two records are returned: employee 100 has a NULL manager, and employee 119 has a NULL department.

### Question 9 ? Hard

**Business scenario:** Job-range compliance checks must identify employees whose salaries fall outside the configured range for their job.

**Problem statement:** Join `employees_df` and `jobs_df` using `col()` expressions on job ID; return employee ID, job ID, salary, minimum salary, and maximum salary where salary is below the job minimum or above the job maximum.

**Expected output:** One employee is outside the configured range: employee 119, job HR_REP, salary 2500.00 against a 4000.00 minimum; all other salaries fall within their job ranges.

## lit()

### Question 10 ? Easy

**Business scenario:** An HR lakehouse migration needs a source label on every employee record.

**Problem statement:** Use `lit()` to add a `source_system` column with value `ORACLE_HR` to `employees_df`.

**Expected output:** All 20 records remain and every row has `source_system` equal to `ORACLE_HR`.

### Question 11 ? Medium

**Business scenario:** A monthly workforce snapshot must carry the same extraction date on employee and department records.

**Problem statement:** Use `lit()` to add `snapshot_date` with value 2025-01-31 to both `employees_df` and `departments_df`; use a date-compatible value and verify its type.

**Expected output:** The employee snapshot has 20 rows, the department snapshot has 18 rows, and both contain a date-typed `snapshot_date` with the same value.

### Question 12 ? Hard

**Business scenario:** The HR migration audit requires a batch identifier and source entity label while retaining rows with missing organizational assignments.

**Problem statement:** Use `lit()` to add `migration_batch_id` and `source_entity` to `employees_df`, then validate the output count and inspect the row for employee 119.

**Expected output:** The output has 20 rows with the two constant metadata columns; employee 119 remains and its `department_id` stays NULL.

## alias()

### Question 13 ? Easy

**Business scenario:** A workforce roster should use business-facing labels for employee identifiers and names.

**Problem statement:** Use `alias()` in a projection to output `employee_id` as `employee_key`, `first_name` as `given_name`, and `last_name` as `family_name`.

**Expected output:** The result has exactly three columns with the requested names and retains all 20 employee rows.

### Question 14 ? Medium

**Business scenario:** Compensation analysts use report terminology different from the source EMPLOYEES column names.

**Problem statement:** Project employee ID, job ID, salary, and commission percentage, applying aliases `employee_key`, `job_code`, `monthly_salary`, and `commission_rate`.

**Expected output:** Four columns are returned in the specified order; salary and commission values retain their source numeric types, including NULL commission values.

### Question 15 ? Hard

**Business scenario:** A department workforce report must distinguish employee manager and department manager identifiers after a join.

**Problem statement:** Join employees to departments on department ID and use qualified columns with `alias()` to return employee ID, employee manager ID, department ID, department name, and department manager ID under unique output names.

**Expected output:** A 20-row result has five unambiguous columns; employee 119 is retained with NULL department name and department manager ID.

## withColumn()

### Question 16 ? Easy

**Business scenario:** A workforce reporting table needs annual salary in addition to the monthly salary.

**Problem statement:** Use `withColumn()` to add `annual_salary` as salary multiplied by 12 on `employees_df`.

**Expected output:** The new column is present for all 20 rows; employee 100 has annual salary 288000.00, and the original salary column remains unchanged.

### Question 17 ? Medium

**Business scenario:** Sales compensation reporting needs the estimated monthly commission amount based on salary and commission percentage.

**Problem statement:** Use `withColumn()` to add `monthly_commission_amount` as salary multiplied by commission percentage. Preserve the existing NULL behavior.

**Expected output:** All 20 rows remain; employee 116 has 1650.00, employee 117 has 800.00, and employees with NULL commission retain NULL commission amounts.

### Question 18 ? Hard

**Business scenario:** HR master-data controls need a prioritized status for records with incomplete hierarchy or contact information.

**Problem statement:** Use `withColumn()` and built-in conditional expressions to add `employee_data_status`: `MISSING_DEPARTMENT` when department ID is NULL, `TOP_LEVEL` when manager ID is NULL, `MISSING_EMAIL` when email is NULL, otherwise `COMPLETE`. Keep the condition order as stated.

**Expected output:** All 20 rows have a status; employee 119 is `MISSING_DEPARTMENT`, employee 100 is `TOP_LEVEL`, and the duplicated FRAHMAN records remain present.

## withColumnRenamed()

### Question 19 ? Easy

**Business scenario:** An HR data consumer requires a clearer name for the employee identifier.

**Problem statement:** Use `withColumnRenamed()` to rename `employee_id` to `employee_key` in `employees_df`.

**Expected output:** The returned DataFrame has `employee_key` instead of `employee_id`, still has 20 rows, and the reusable source DataFrame keeps its original column name.

### Question 20 ? Medium

**Business scenario:** The organization hierarchy feed uses canonical reporting names for department and manager keys.

**Problem statement:** Use `withColumnRenamed()` to rename `department_id` to `org_department_id` and `manager_id` to `reports_to_employee_id` in `employees_df`; validate the new names and row count.

**Expected output:** The two source names are replaced by the requested names, all other columns remain, and all 20 rows are retained.

### Question 21 ? Hard

**Business scenario:** A target HR model joins employee and department data using a consistently named organization key.

**Problem statement:** Rename the department key in `employees_df` and `departments_df` to a common target name, then left join on that key. Confirm employee and department manager columns remain distinguishable and missing department assignments are retained.

**Expected output:** The result has 20 employee rows, one shared target department key, separate employee/department manager fields, and a NULL department match for employee 119.

## withColumns()

### Question 22 ? Easy

**Business scenario:** A workforce snapshot needs source and run-date metadata added together to the employee master.

**Problem statement:** Use `withColumns()` to add constant `source_system` and date-typed `snapshot_date` fields to `employees_df` in a single call.

**Expected output:** The resulting DataFrame has both metadata columns, all original fields, and 20 rows.

### Question 23 ? Medium

**Business scenario:** HR cost reporting needs monthly and annual salary measures generated together for each employee.

**Problem statement:** Use `withColumns()` to add `monthly_salary_copy` from salary and `annual_salary` as salary multiplied by 12.

**Expected output:** The two added columns appear alongside the source fields; there are 20 rows and employee 100 has 24000.00 monthly and 288000.00 annual salary.

### Question 24 ? Hard

**Business scenario:** The HR quality dashboard needs independent flags for missing department, missing manager, and missing commission data.

**Problem statement:** Use one `withColumns()` call to add Boolean fields `missing_department`, `missing_manager`, and `missing_commission` using the respective NULL checks. Validate the true counts for each flag.

**Expected output:** The flags are Boolean; `missing_department` is true once, `missing_manager` is true once, and `missing_commission` is true 17 times. The source DataFrame is unchanged.

## drop()

### Question 25 ? Easy

**Business scenario:** A workforce directory export should omit employee phone numbers.

**Problem statement:** Use `drop()` to remove `phone_number` from `employees_df`.

**Expected output:** The output no longer has `phone_number`, retains all other source columns, and contains 20 rows.

### Question 26 ? Medium

**Business scenario:** An HR analytics product must remove direct contact details and tolerate a legacy field that is not present in the current schema.

**Problem statement:** Use `drop()` to remove `email`, `phone_number`, and `legacy_badge_number` from `employees_df`; inspect the resulting columns and count.

**Expected output:** Email and phone number are absent, dropping the nonexistent legacy field does not fail, and the output retains 20 rows.

### Question 27 ? Hard

**Business scenario:** A joined workforce report contains employee and department manager fields with the same source name; the export must remove both manager identifiers.

**Problem statement:** Join `employees_df` and `departments_df` on department ID, then use `drop()` to remove the ambiguous `manager_id` field(s). Validate that neither manager field remains while other joined fields and employee rows are preserved.

**Expected output:** No `manager_id` column remains in the output, department and employee descriptive fields are retained, and the left-join output still has 20 rows including employee 119.

## transform()

### Question 28 ? Easy

**Business scenario:** An employee directory pipeline needs one reusable transformation to prepare a compact employee roster.

**Problem statement:** Define a DataFrame transformation function that returns employee ID, first name, last name, and job ID; apply it to `employees_df` with `transform()`.

**Expected output:** The transformed result contains exactly those four columns and all 20 employee records.

### Question 29 ? Medium

**Business scenario:** A compensation export applies the same two preparation steps to every HR extract: select workforce fields, then add annual salary.

**Problem statement:** Define reusable DataFrame transformation functions for the projection and annual-salary addition, then compose them with `transform()` on `employees_df`.

**Expected output:** The final result has employee ID, job ID, salary, and annual salary; it contains 20 rows and employee 100 annualizes to 288000.00.

### Question 30 ? Hard

**Business scenario:** The HR migration pipeline needs a repeatable job-range validation view without mutating its reusable source DataFrames.

**Problem statement:** Define a transformation function that joins employee rows to `jobs_df` by job ID, projects employee ID, job ID, salary, minimum salary, and maximum salary, and retains employees outside the configured range. Apply the function with `transform()` and verify the source DataFrames remain unchanged.

**Expected output:** One row is returned for employee 119 with HR_REP salary 2500.00 and minimum salary 4000.00; the source employee and job DataFrames retain their original schemas and row counts.

## 7. Final practice completion checklist

- [ ] I completed all 30 question cells independently.
- [ ] I verified column names, data types, row counts, and NULL behavior after transformations.
- [ ] I preserved HR keys and did not silently remove duplicate business identifiers.
- [ ] I confirmed the reusable source DataFrames remain unchanged by transformations.

After completing your work, stop Spark with `spark.stop()` in your own cell.